# nexus-lite kernel primitives on Colab

Runnable tour of the core `nexus-lite` v0.3.x pure-Python stdlib kernel primitives: (1) 3-tier memory (`stm`/`mtm`/`ltm`), (2) Bus v3 monotonic `seq` + multi-consumer cursors (`consumer=`), (3) dependency-free PTY controller (`nexus_lite.pty`), (4) code-tallied `ballot` voting + `cas` blobs + `#nexus.ledger/2` step ledgers, (5) `NEXUS_SENSITIVITY=phi` gate with `stm/1` redaction, and (6) zero-preinstall remote compute (`run_local_sandbox` / `run_on_colab`). No GPU, no API keys.

## 1. Install

In [ ]:
!pip install -qU nexus-lite

## 2. Memory tiers: STM, MTM, LTM
Everything lives in an isolated scratch workspace (`NEXUS_LITE_DIR`). MTM flow checkpoints are bus messages of `kind='checkpoint'`; the bus is the MTM transport.

In [ ]:
import os, pathlib, shutil, tempfile, time
os.environ.pop('NEXUS_SENSITIVITY', None)
os.environ['NEXUS_LITE_DIR'] = tempfile.mkdtemp(prefix='nexus-prims-')
from nexus_lite import _db
_db.reset_init()
from nexus_lite.memory import stm, mtm, ltm

# STM: file-backed stm/1 nodes
node = stm.stm_write(kind='task', status='running', summary='segment lung nodules', agent='worker-1')
print('stm_read   :', stm.stm_read(node['id'])['summary'])
print('stm_pack   :', stm.stm_pack()['schema'], 'nodes =', stm.stm_pack()['node_count'])

# MTM: flow checkpoint + state carried over the bus
cp = mtm.bus_send('orchestrator', 'step 2/5 done: features extracted', sender='worker-1', kind='checkpoint')
print('mtm state  :', [m['summary'] for m in mtm.bus_read('orchestrator') if m['kind'] == 'checkpoint'])

# LTM: markdown wiki + FTS5 search
ltm.wiki_write('notes/radiomics', '# Radiomics\nIBSI first-order features need discretisation.')
print('ltm search :', [h['slug'] for h in ltm.wiki_search('discretisation')])

## 3. Bus v3: monotonic `seq`, multi-consumer cursors, offline redelivery, prune

In [ ]:
m1 = mtm.bus_send('bob', 'first', sender='al')
m2 = mtm.bus_send('bob', 'second', sender='al')
print('monotonic seq:', m1['seq'], '<', m2['seq'])

# bob was offline when the messages were sent; registering redelivers them
print('redelivered  :', mtm.bus_register('bob')['redelivered'] == [m1['id'], m2['id']])

# Independent per-consumer cursors: advancing 'auditor' leaves 'indexer' untouched
mtm.bus_ack('bob', m1['seq'], consumer='auditor')
print('auditor left :', [m['summary'] for m in mtm.bus_read('bob', unacked=True, consumer='auditor')])
print('indexer left :', [m['summary'] for m in mtm.bus_read('bob', unacked=True, consumer='indexer')])

# Once all active consumers ack m1, bus_prune deletes m1 while keeping unacked m2
mtm.bus_ack('bob', m1['seq'], consumer='indexer')
time.sleep(0.01)
print('bus_prune    :', mtm.bus_prune(0), 'deleted')
print('unacked left :', [m['summary'] for m in mtm.bus_read('bob', unacked=True, consumer='auditor')])

## 4. Persistent PTY sessions (`nexus_lite.pty`, no `dtach` binary required)

In [ ]:
from nexus_lite import pty

sock_dir = tempfile.mkdtemp(prefix='nxs-', dir='/tmp')
os.environ['NEXUS_LITE_SOCK_DIR'] = sock_dir
sess = pty.spawn('cat', name='demo-pty')
try:
    pty.send(sess, 'stdlib-pty-ready')
    screen_txt = ''
    for _ in range(30):
        screen_txt = pty.read(sess, settle=0.05)
        if 'stdlib-pty-ready' in screen_txt:
            break
        time.sleep(0.1)
    active = [s['name'] for s in pty.list_sessions()]
    print('pty output :', 'stdlib-pty-ready' in screen_txt, '| active:', active)
    assert 'stdlib-pty-ready' in screen_txt
finally:
    pty.kill_session(sess)
    shutil.rmtree(sock_dir, ignore_errors=True)
    os.environ.pop('NEXUS_LITE_SOCK_DIR', None)

## 5. Code-tallied `ballot` voting, content-addressed `cas`, and `#nexus.ledger/2`

In [ ]:
from nexus_lite import ballot, cas, ledger

art = pathlib.Path(os.environ['NEXUS_LITE_DIR']) / 'ibsi_check.txt'
art.write_text('270/270 pass\n', encoding='utf-8')

ballot.open_ballot('ibsi-gate', 'Approve IBSI parity?', ['arch', 'phys', 'stat'], opener='lead', attach=[str(art)])
ballot.cast('ibsi-gate', 'arch', 'yes')
st = ballot.cast('ibsi-gate', 'phys', 'yes')
print('ballot     :', st['decided'], f"({st['yes']} yes / {st['no']} no)")
assert st['decided'] == 'yes'

sha = cas.put(art)
assert cas.get(sha) == b'270/270 pass\n'
print('cas sha256 :', sha[:16])

led = pathlib.Path(os.environ['NEXUS_LITE_DIR']) / 'steps.ledger'
ledger.append(led, 'ibsi_verify', 'OK', 0.15, '-', str(art), 'PASS')
print('ledger rows:', len(ledger.rows(led)))

## 6. PHI sensitivity gate
With `NEXUS_SENSITIVITY=phi` only local runners (local Ollama, shell, python) may be dispatched; cloud CLIs are refused. `stm/1` summaries are redacted in the packed brief.

In [ ]:
from nexus_lite.phi import phi_gate_error
os.environ['NEXUS_SENSITIVITY'] = 'phi'
for cmd in ['claude', 'agy -p', 'ollama launch claude', 'ollama run llama3.2']:
    err = phi_gate_error(cmd)
    print(f'{cmd!r:28} ->', 'BLOCKED' if err else 'allowed (local)')

stm.stm_write(kind='task', summary='patient John Doe MRN 12345')
brief = stm.stm_pack(phi=True)
print('sensitivity:', brief['sensitivity'], '| headline:', brief['headline'])
print('summaries  :', [n['summary'] for n in brief['nodes']])
assert all(n['summary'] == '' for n in brief['nodes'])
os.environ.pop('NEXUS_SENSITIVITY')

## 7. Zero-preinstall remote compute
`pack_self_contained_script` builds one self-extracting `.py` that carries `nexus_lite`; `run_local_sandbox` runs it in an isolated `python -I` process and `run_on_colab` ships it to a runtime over `POST /exec`.

In [ ]:
import nexus_lite.compute as compute
packed = compute.pack_self_contained_script(python='RESULT = 6 * 7')
print('packed script:', len(packed[0]), 'bytes')
sb = compute.run_local_sandbox(python='RESULT = {"sandbox": 6 * 7}')
print('local sandbox:', sb['result'])

from nexus_lite.colab import tunnel as colab_tunnel
info = colab_tunnel.start_tunnel(port=0, tunnel=False)
try:
    tok = colab_tunnel.token_file_path(info['port']).read_text().strip()
    res = compute.run_on_colab(python='RESULT = {"answer": 6 * 7}', url=info['local_url'], token=tok)
    print('run_on_colab result:', res['result'])
finally:
    colab_tunnel.stop_tunnel()